# Data: cases, splits and the AMR graph

**Data.** PRISMS-PF dendritic solidification on the solver's adaptive (AMR) mesh, 301 frames per case (t = 0..300), stored in 30 epochs of 10 frames (one mesh per epoch). Parameters: undercooling delta, anisotropy epsilonM, diffusivity D. All cases and their sets are listed in `cases.csv`.

| Set | Cases | Use |
| --- | ---: | --- |
| train | 100 = 80 main LHS cases + 20 high-undercooling cases (`hd_*`, delta 0.80-0.90) | training |
| val | 10 | every model and threshold choice, always all 10 |
| test | 10 | final evaluation only |
| outer | 20 (outside the training box in one parameter) | final evaluation only |

The split file `splits.json` is the only split used in this folder; do not reshuffle cases elsewhere. `flags.near_boundary_train` lists hd_13, hd_17 and hd_19, whose dendrite comes within 41-59 length units of the domain boundary at t = 300.

**Graph of one AMR epoch** (`amr/data.py`): nodes are the FE_Q(3) Gauss-Lobatto nodes of every cell (4 x 4 per cell, so a node value is the field value at that point); edges link lattice neighbours along x, y and both diagonals, plus each hanging node to the 4 nodes of the edge it lies on; all edges in both directions.

This notebook checks the split, shows the parameter coverage, and checks that the graph code rebuilds the cached graphs exactly. Run it through HTCondor, not on the access point.

In [ ]:
cache_dir = 'cache'
cases_path = 'cases.csv'
split_path = 'splits.json'
out_dir = 'outputs/data'
check_epochs = [0, 29]

In [ ]:
import json
import os
import sys
import time

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, os.getcwd())
from amr.data import PARAM_BOX, build_graph, load_epoch, load_splits

os.makedirs(out_dir, exist_ok=True)
PARAMS = ['delta', 'epsilonM', 'D']
cases = pd.read_csv(cases_path).set_index('case')
split = load_splits(split_path)
names = ['train', 'val', 'test', 'outer']

## 1. Split check

Stops the notebook if a case is missing, duplicated, in two sets, or in a different set than `cases.csv` says.

In [ ]:
for name in names:
    assert len(set(split[name])) == len(split[name]), f'duplicate case in {name}'
    assert sorted(split[name]) == sorted(cases.index[cases['set'] == name]), f'{name} differs from cases.csv'
for i, a in enumerate(names):
    for b in names[i + 1:]:
        assert not set(split[a]) & set(split[b]), f'{a} and {b} share cases'
assert set(split['flags']['near_boundary_train']) <= set(split['train'])
print({name: len(split[name]) for name in names})
print(cases.groupby(['set', 'campaign']).size())

## 2. Parameter coverage

`nearest_train` is the distance from a held-out case to the closest training case, with each parameter scaled to [0, 1] over the training box. Larger values mean the case is harder to interpolate.

In [ ]:
def scaled(names_):
    p = cases.loc[names_, PARAMS].to_numpy(float)
    return (p - PARAM_BOX[:, 0]) / (PARAM_BOX[:, 1] - PARAM_BOX[:, 0])

train_scaled = scaled(split['train'])
rows = []
for name in ('val', 'test', 'outer'):
    for case, x in zip(split[name], scaled(split[name])):
        d = np.linalg.norm(train_scaled - x, axis=1)
        rows.append({'set': name, 'case': case, **cases.loc[case, PARAMS].to_dict(),
                     'inside_box': bool(((x >= 0) & (x <= 1)).all()),
                     'nearest_train': float(d.min()), 'nearest_train_case': split['train'][int(d.argmin())]})
coverage = pd.DataFrame(rows)
coverage.to_csv(os.path.join(out_dir, 'heldout_coverage.csv'), index=False)
print(pd.concat({n: cases.loc[split[n], PARAMS].agg(['min', 'max']) for n in names}))
print(coverage.groupby('set')['nearest_train'].describe())

colors = {'train': '#9aa5b1', 'val': '#1f77b4', 'test': '#d62728', 'outer': '#2ca02c'}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (i, j) in zip(axes, [(0, 1), (0, 2), (1, 2)]):
    for name in names:
        p = cases.loc[split[name], PARAMS].to_numpy(float)
        ax.scatter(p[:, i], p[:, j], s=18, c=colors[name], label=name)
    ax.set_xlabel(PARAMS[i])
    ax.set_ylabel(PARAMS[j])
axes[0].legend()
fig.suptitle('Case parameters by set')
fig.tight_layout()
fig.savefig(os.path.join(out_dir, 'split_parameters.png'), dpi=120)

## 3. Graph check and graph sizes

Rebuild the graph of every case at the first and last epoch from the mesh alone (`pos`, `cells`) and compare it with the cached graph. `edges_equal`, `node_type_equal` and `h_equal` must all be True. `build_seconds` is the cost remeshing pays after every remesh.

In [ ]:
rows, start = [], time.time()
for case in cases.index:
    for e in check_epochs:
        d = load_epoch(cache_dir, case, e)
        t0 = time.perf_counter()
        ei, node_type, h = build_graph(d['pos'].astype(np.float64), d['cells'].astype(np.int64))
        seconds = time.perf_counter() - t0
        cached = d['edge_index'].astype(np.int64)
        rows.append({'case': case, 'set': cases.loc[case, 'set'], 'epoch': e, 'nodes': len(d['pos']),
                     'edges': cached.shape[1],
                     'edges_equal': ei.shape == cached.shape and bool((ei == cached).all()),
                     'node_type_equal': bool((node_type == d['node_type']).all()),
                     'h_equal': bool(np.allclose(h, d['h'], rtol=1e-5)), 'build_seconds': seconds})
graphs = pd.DataFrame(rows)
graphs.to_csv(os.path.join(out_dir, 'graph_check.csv'), index=False)
mismatch = graphs[~(graphs['edges_equal'] & graphs['node_type_equal'] & graphs['h_equal'])]
print(f'{len(graphs)} graphs rebuilt in {time.time() - start:.0f} s, {len(mismatch)} mismatches')
print(graphs.groupby(['set', 'epoch'])[['nodes', 'edges', 'build_seconds']].agg(['median', 'max']))

summary = {'sizes': {n: len(split[n]) for n in names},
           'nearest_train_max': coverage.groupby('set')['nearest_train'].max().to_dict(),
           'graphs_rebuilt': int(len(graphs)), 'graph_mismatches': int(len(mismatch)),
           'build_seconds_median': float(graphs['build_seconds'].median()),
           'largest_graph': {'nodes': int(graphs['nodes'].max()), 'edges': int(graphs['edges'].max())}}
with open(os.path.join(out_dir, 'summary.json'), 'w') as fh:
    json.dump(summary, fh, indent=1)
print(json.dumps(summary, indent=1))
assert len(mismatch) == 0